# Stream C - Stance Scoring

Computes caption/evidence similarity features. Uses GPU with SentenceTransformers if available, otherwise TF-IDF fallback.


In [1]:
from pathlib import Path
import sys, os, json, pickle, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import STREAM_C_CACHE, STREAM_A_CACHE, STREAM_B_CACHE, ALL_MANIFEST, FIGURES, TABLES

STREAM_C_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("STREAM_C_CACHE:", STREAM_C_CACHE)


ROOT: E:\eve-rag
STREAM_C_CACHE: E:\eve-rag\cache\stream_c


In [2]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
BATCH_SIZE = 256


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM GB: 8.0


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
EVIDENCE_DIR = STREAM_C_CACHE / "evidence"; STANCE_DIR = STREAM_C_CACHE / "stance_scores"; STANCE_DIR.mkdir(parents=True, exist_ok=True)
evidence_path = EVIDENCE_DIR / "raw_evidence_all.csv"
if not evidence_path.exists(): raise FileNotFoundError(f"Missing {evidence_path}. Run evidence retrieval first.")
evidence = pd.read_csv(evidence_path, dtype={"pair_id":"string","caption":"string","evidence_text":"string"}, low_memory=False)
print("Evidence rows:", len(evidence)); display(evidence.head())


Evidence rows: 97361


,pair_id,dataset,split,label,caption,evidence_text,evidence_len,caption_len
0,nc_620701_0,newsclippings,train,0,Saudi troops cheer as they ride at the back of...,Caption: Saudi troops cheer as they ride at th...,387,125
1,nc_620701_1,newsclippings,train,1,Israeli soldiers ride on a tank to a position ...,Caption: Israeli soldiers ride on a tank to a ...,328,73
2,nc_1569820_0,newsclippings,train,0,Tiffany is north east of Aberdeen,Caption: Tiffany is north east of Aberdeen | A...,325,33
3,nc_1569820_1,newsclippings,train,1,Some ferry services were affected between Holy...,Caption: Some ferry services were affected bet...,358,71
4,nc_1490080_0,newsclippings,train,0,Rock band Paramore is another group which sell...,Caption: Rock band Paramore is another group w...,349,55


In [4]:
USE_SENTENCE_TRANSFORMERS = True
model = None
try:
    from sentence_transformers import SentenceTransformer
    MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    print("Using SentenceTransformer:", MODEL_NAME, "device:", DEVICE)
except Exception as e:
    USE_SENTENCE_TRANSFORMERS = False
    print("SentenceTransformers unavailable; using TF-IDF fallback.", repr(e))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2 device: cuda


In [5]:
captions = evidence["caption"].fillna("").astype(str).tolist(); evtexts = evidence["evidence_text"].fillna("").astype(str).tolist()
if USE_SENTENCE_TRANSFORMERS:
    cap_emb = model.encode(captions, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    ev_emb = model.encode(evtexts, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    sim = np.sum(cap_emb * ev_emb, axis=1)
else:
    vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
    X = vec.fit_transform(captions + evtexts); cap_X = X[:len(captions)]; ev_X = X[len(captions):]
    sim = np.asarray(cap_X.multiply(ev_X).sum(axis=1)).ravel()


Batches:   0%|          | 0/381 [00:00<?, ?it/s]

Batches:   0%|          | 0/381 [00:00<?, ?it/s]

In [6]:
df = evidence[["pair_id", "dataset", "split", "label", "caption", "evidence_text"]].copy()
df["stance_similarity"] = sim.astype(float)
df["caption_len"] = df["caption"].fillna("").astype(str).str.len(); df["evidence_len"] = df["evidence_text"].fillna("").astype(str).str.len()
df["len_ratio"] = df["caption_len"] / (df["evidence_len"] + 1)
df["support_score"] = df["stance_similarity"]; df["contradiction_score"] = 1.0 - df["stance_similarity"]; df["neutral_score"] = 1.0 - np.abs(df["stance_similarity"] - np.nanmean(df["stance_similarity"]))
df.to_csv(STANCE_DIR / "stream_c_stance_scores_all.csv", index=False)
pickle.dump(df, open(STANCE_DIR / "stream_c_stance_scores_all.pkl", "wb"), protocol=4)
for (dataset, split), sub in df.groupby(["dataset", "split"]): sub.to_csv(STANCE_DIR / f"stream_c_stance_scores_{dataset}_{split}.csv", index=False)
summary = df.groupby(["dataset", "split"]).agg(rows=("pair_id", "count"), sim_mean=("stance_similarity", "mean"), sim_std=("stance_similarity", "std")).reset_index()
summary.to_csv(TABLES / "stream_c_stance_summary.csv", index=False); display(summary)
print("Stance scoring complete.")


,dataset,split,rows,sim_mean,sim_std
0,mmfakebench,test,10000,0.663462,0.140448
1,mmfakebench,val,1000,0.675598,0.137493
2,newsclippings,test,7264,0.815615,0.068803
3,newsclippings,train,71072,0.813873,0.068167
4,newsclippings,val,7024,0.816777,0.068392
5,verite,test,1001,0.917001,0.036142


Stance scoring complete.
